In [10]:
"""
SHAP (SHapley Additive exPlanations) Analysis
==============================================
Explain individual predictions by showing which spectral features
(ppm regions) contributed most to each classification decision.

Install required package first:
    pip install shap

This may take a few minutes to compute SHAP values for the first time.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import json
import shap
from pathlib import Path
import os

# Create output directory
OUTPUT_DIR = Path("shap_output")
OUTPUT_DIR.mkdir(exist_ok=True)

print("="*60)
print("SHAP EXPLAINABILITY ANALYSIS")
print("="*60)
print(f"Output directory: {OUTPUT_DIR}/")
print("="*60)

# --- 1. LOAD DATA AND MODEL ---
print("\n[1] Loading data and model...")

# Load the data (you'll need to run this after your RF training notebook)
# Assuming these variables are available from your training notebook
# If running as standalone, you'll need to reload the data

df = pd.read_csv("processed_data/nmr_features_with_groups.csv")
X = df.iloc[:, 2:].values
y = df['group'].values

# Load saved model
MODEL_DIR = Path("models")
with open(MODEL_DIR / "rf_tuned_model.pkl", 'rb') as f:
    rf_model = pickle.load(f)

with open(MODEL_DIR / "label_encoder.pkl", 'rb') as f:
    le = pickle.load(f)

with open(MODEL_DIR / "preprocessing_params.json", 'r') as f:
    params = json.load(f)

print(f"✓ Model loaded: {rf_model.__class__.__name__}")
print(f"✓ Data shape: {X.shape}")
print(f"✓ Classes: {le.classes_}")

# Get bin centers for ppm mapping
bin_centers = np.linspace(params['ppm_min'], params['ppm_max'], params['n_bins'])

# --- 2. PREPARE DATA ---
print("\n[2] Preparing train/test split...")

# Recreate the same split as training (same random seed)
from sklearn.model_selection import train_test_split

y_encoded = le.transform(y)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_encoded, test_size=0.30, random_state=42, stratify=y_encoded
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")

# --- 3. CREATE SHAP EXPLAINER ---
print("\n[3] Creating SHAP explainer...")
print("This may take 1-2 minutes for Random Forest...")

# Use TreeExplainer for Random Forest (faster than KernelExplainer)
# Use a sample of training data as background for efficiency
background_sample = shap.sample(X_train, 100)  # Use 100 samples as background
explainer = shap.TreeExplainer(rf_model, background_sample)

print("✓ Explainer created")

# --- 4. COMPUTE SHAP VALUES ---
print("\n[4] Computing SHAP values for test set...")
print("This will take a few minutes...")

# Compute SHAP values for test set
shap_values = explainer.shap_values(X_test)

# Check the format and fix if needed
if isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
    # Wrong format: (n_samples, n_features, n_classes)
    # Need to convert to list of arrays: [class0_array, class1_array, ...]
    print("Converting SHAP values to correct format...")
    n_samples, n_features, n_classes = shap_values.shape
    shap_values = [shap_values[:, :, i] for i in range(n_classes)]

# Verify correct format
print(f"✓ SHAP values computed")
if isinstance(shap_values, list):
    print(f"   Shape: {len(shap_values)} classes, each with shape {shap_values[0].shape}")
else:
    print(f"   WARNING: Unexpected shape: {shap_values.shape}")

# --- 5. VISUALIZATIONS ---
print("\n[5] Creating SHAP visualizations...")

# 5.1 Summary Plot - Overall feature importance across all classes
fig, ax = plt.subplots(figsize=(14, 8))

# Create feature names with ppm values
feature_names_ppm = [f"{ppm:.2f}" for ppm in bin_centers]

shap.summary_plot(shap_values, X_test, 
                  feature_names=feature_names_ppm,
                  class_names=le.classes_, 
                  show=False, 
                  max_display=20,
                  plot_size=(14, 8))
plt.xlabel('SHAP value (impact on model output)', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'shap_summary_plot.png', dpi=300, bbox_inches='tight')
print(f"✓ Saved: {OUTPUT_DIR / 'shap_summary_plot.png'}")
plt.close()

# 5.2 Mean absolute SHAP values per class (bar plot)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, class_name in enumerate(le.classes_):
    ax = axes[i]
    
    # Get mean absolute SHAP values for this class
    mean_shap = np.abs(shap_values[i]).mean(axis=0)
    
    # Get top 15 features
    top_indices = np.argsort(mean_shap)[-15:]
    top_values = mean_shap[top_indices]
    top_ppms = bin_centers[top_indices]
    
    # Plot
    ax.barh(range(len(top_indices)), top_values, color='steelblue', edgecolor='black', linewidth=0.5)
    ax.set_yticks(range(len(top_indices)))
    ax.set_yticklabels([f"{ppm:.3f} ppm" for ppm in top_ppms], fontsize=9)
    ax.set_xlabel('Mean |SHAP value|', fontsize=10)
    ax.set_title(f'{class_name}', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')

# Remove extra subplot
axes[-1].remove()

plt.tight_layout()
plt.savefig('shap_output/shap_per_class_importance.png', dpi=300, bbox_inches='tight')
print("✓ Saved: shap_output/shap_per_class_importance.png")
plt.close()

# --- 6. INDIVIDUAL PREDICTION EXPLANATIONS ---
print("\n[6] Creating example individual explanations...")

# Select a few interesting test samples
n_examples = 5
example_indices = np.random.choice(len(X_test), n_examples, replace=False)

for idx in example_indices:
    sample = X_test[idx]
    true_class = le.inverse_transform([y_test[idx]])[0]
    pred_class_encoded = rf_model.predict([sample])[0]
    pred_class = le.inverse_transform([pred_class_encoded])[0]
    pred_proba = rf_model.predict_proba([sample])[0]
    confidence = pred_proba[pred_class_encoded]
    
    print(f"\nExample {idx}:")
    print(f"  True class: {true_class}")
    print(f"  Predicted: {pred_class} ({confidence:.1%} confidence)")
    
    # Force plot for the predicted class
    fig, ax = plt.subplots(figsize=(14, 3))
    
    # Get SHAP values for predicted class
    shap_vals_for_pred = shap_values[pred_class_encoded][idx]
    
    # Find top contributing features
    top_n = 10
    top_indices = np.argsort(np.abs(shap_vals_for_pred))[-top_n:][::-1]
    
    print(f"  Top contributing ppm regions:")
    for i in top_indices[:5]:
        contribution = shap_vals_for_pred[i]
        direction = "toward" if contribution > 0 else "against"
        print(f"    {bin_centers[i]:.3f} ppm: {contribution:+.4f} ({direction} {pred_class})")
    
    # Waterfall plot
    shap_explanation = shap.Explanation(
        values=shap_vals_for_pred,
        base_values=explainer.expected_value[pred_class_encoded],
        data=sample,
        feature_names=[f"{ppm:.2f} ppm" for ppm in bin_centers]
    )
    
    shap.plots.waterfall(shap_explanation, max_display=15, show=False)
    plt.title(f'SHAP Explanation: {pred_class} (True: {true_class})', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'shap_waterfall_example_{idx}.png', dpi=300, bbox_inches='tight')
    plt.close()

print(f"\n✓ Saved {n_examples} waterfall plots")

# --- 7. AGGREGATE ANALYSIS ---
print("\n[7] Aggregate SHAP analysis...")

# For each class, find which ppm regions are most important
print("\nMost important ppm regions per class:")
for i, class_name in enumerate(le.classes_):
    mean_abs_shap = np.abs(shap_values[i]).mean(axis=0)
    top_5_idx = np.argsort(mean_abs_shap)[-5:][::-1]
    
    print(f"\n{class_name}:")
    for rank, idx in enumerate(top_5_idx, 1):
        print(f"  {rank}. {bin_centers[idx]:.3f} ppm (SHAP: {mean_abs_shap[idx]:.5f})")

# --- 8. SAVE SHAP VALUES ---
print("\n[8] Saving SHAP values for future use...")

shap_data = {
    'shap_values': [sv.tolist() for sv in shap_values],  # Convert to list for JSON
    'test_indices': list(range(len(X_test))),
    'feature_names': [f"{ppm:.3f}" for ppm in bin_centers],
    'class_names': le.classes_.tolist()
}

with open(OUTPUT_DIR / 'shap_values.json', 'w') as f:
    json.dump(shap_data, f)

print(f"✓ Saved: {OUTPUT_DIR / 'shap_values.json'}")

print("\n" + "="*60)
print("SHAP ANALYSIS COMPLETE")
print("="*60)
print(f"\nAll files saved to: {OUTPUT_DIR}/")
print("\nGenerated files:")
print(f"  1. {OUTPUT_DIR / 'shap_summary_plot.png'} - Overall feature importance")
print(f"  2. {OUTPUT_DIR / 'shap_per_class_importance.png'} - Top features per class")
print(f"  3. {OUTPUT_DIR / 'shap_waterfall_example_*.png'} - Individual predictions")
print(f"  4. {OUTPUT_DIR / 'shap_values.json'} - SHAP values for future use")
print("\nNext steps:")
print("  - Add these visualizations to your report")
print("  - Integrate SHAP explanations into the Streamlit app (optional)")
print("  - Compare SHAP importance with RF feature importance")
print("="*60)

SHAP EXPLAINABILITY ANALYSIS
Output directory: shap_output/

[1] Loading data and model...
✓ Model loaded: RandomForestClassifier
✓ Data shape: (847, 400)
✓ Classes: ['Aromatics' 'Heterocycles & nucleotides' 'Lipids'
 'Nitrogenous & organic acids' 'Organic oxygen compounds']

[2] Preparing train/test split...
Train: 592, Val: 127, Test: 128

[3] Creating SHAP explainer...
This may take 1-2 minutes for Random Forest...
✓ Explainer created

[4] Computing SHAP values for test set...
This will take a few minutes...


c:\Users\nilsa\GitHub\DDLS_project\code\venv\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.4.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\nilsa\GitHub\DDLS_project\code\venv\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.4.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\nilsa\GitHub\DDLS_project\code\venv\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LabelEncode

Converting SHAP values to correct format...
✓ SHAP values computed
   Shape: 5 classes, each with shape (128, 400)

[5] Creating SHAP visualizations...
✓ Saved: shap_output\shap_summary_plot.png
✓ Saved: shap_output/shap_per_class_importance.png

[6] Creating example individual explanations...

Example 19:
  True class: Heterocycles & nucleotides
  Predicted: Nitrogenous & organic acids (37.4% confidence)
  Top contributing ppm regions:
    3.634 ppm: +0.0088 (toward Nitrogenous & organic acids)
    2.381 ppm: -0.0062 (against Nitrogenous & organic acids)
    3.784 ppm: +0.0054 (toward Nitrogenous & organic acids)
    3.659 ppm: +0.0053 (toward Nitrogenous & organic acids)
    7.318 ppm: +0.0050 (toward Nitrogenous & organic acids)

Example 47:
  True class: Nitrogenous & organic acids
  Predicted: Nitrogenous & organic acids (31.7% confidence)
  Top contributing ppm regions:
    4.035 ppm: -0.0251 (against Nitrogenous & organic acids)
    3.860 ppm: +0.0159 (toward Nitrogenous & organ